In [1]:
from datasets import load_dataset

ds = load_dataset("google-research-datasets/conceptual_captions", "labeled")

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/178M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/178M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/178M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2007090 [00:00<?, ? examples/s]

In [8]:
ds

DatasetDict({
    train: Dataset({
        features: ['image_url', 'caption', 'labels', 'MIDs', 'confidence_scores'],
        num_rows: 2007090
    })
})

In [9]:
from datasets import load_dataset
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import requests
from PIL import Image
from io import BytesIO
import json
from tqdm import tqdm

def download_and_cache_dataset(output_dir="onceptual_captions_data", max_workers=32):
    """Pre-download all images to disk"""
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    # Load dataset
    print("Loading dataset...")
    ds = load_dataset("google-research-datasets/conceptual_captions", "unlabeled")
    train_ds = ds['train']
    total_images = len(train_ds)

    def download_image(args):
        idx, sample = args
        img_path = output_dir / f"{idx:08d}.jpg"

        if img_path.exists():
            return True

        try:
            response = requests.get(sample['image_url'], timeout=5)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content)).convert('RGB')
                img.save(img_path)
                return True
        except: # Catch specific exceptions if possible, or generic for robustness here
            pass
        return False

    # Download train split
    print(f"Downloading {total_images} training images with {max_workers} workers...")

    # Use a list of tuples for mapping to avoid lambda issues in some environments
    # and to make passing arguments cleaner for the executor
    tasks = list(enumerate(train_ds))

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Use tqdm to wrap the immediate iterator from executor.map
        # Using imap_unordered or similar might be slightly more efficient for just progress,
        # but map ensures order if needed (though not strictly needed here for just counting).
        # For a simple progress bar on completion:
        for result in tqdm(executor.map(download_image, tasks), total=total_images, desc="Downloading"):
             results.append(result)

    success_count = sum(results)
    print(f"Successfully downloaded {success_count}/{total_images} images")

    # Save metadata
    print("Saving metadata...")
    metadata = {
        'total': total_images,
        'successful': success_count,
        # Warning: This might be very large to keep in memory all at once if dataset is huge
        'captions': train_ds['caption']
    }
    with open(output_dir / "metadata.json", 'w') as f:
        json.dump(metadata, f)
    print("Done.")

if __name__ == '__main__':
    download_and_cache_dataset()

Loading dataset...


train-00000-of-00002.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.77M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3318333 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15840 [00:00<?, ? examples/s]

Downloading:   1%|          | 37722/3318333 [10:14<9:17:44, 98.03it/s]  

In [1]:
from datasets import load_dataset
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import requests
from PIL import Image
from io import BytesIO
import json
from tqdm import tqdm

def download_and_cache_dataset(output_dir="conceptual_captions_data", split="train", max_workers=32):
    root_dir = Path(output_dir)
    images_dir = root_dir / split
    images_dir.mkdir(parents=True, exist_ok=True)

    print(f"Loading '{split}' dataset from Hugging Face...")
    ds = load_dataset("google-research-datasets/conceptual_captions", split=split)
    
    # --- SAVE ANNOTATIONS FIRST ---
    anno_path = root_dir / f"{split}.jsonl"
    print(f"Saving ALL annotations immediately to {anno_path}...")
    
    with open(anno_path, 'w') as f:
        # Iterate through the dataset directly to save metadata before downloading
        for idx, sample in enumerate(tqdm(ds, desc="Writing JSONL")):
            filename = f"{idx:08d}.jpg"
            relative_path = f"{split}/{filename}"
            # We save what the filename WILL be. 
            # NOTE: Some of these will fail to download later.
            entry = {"file_name": relative_path, "text": sample['caption']}
            f.write(json.dumps(entry) + '\n')
            
    print(f"Annotations saved. Starting download of {len(ds)} images...")
    # -----------------------------

    def download_task(args):
        idx, sample = args
        filename = f"{idx:08d}.jpg"
        img_path = images_dir / filename
        
        if img_path.exists():
             return True

        try:
            response = requests.get(sample['image_url'], timeout=5)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content)).convert('RGB')
                img.save(img_path)
                return True
        except:
            pass
        return False

    tasks = list(enumerate(ds))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(download_task, tasks), total=len(tasks), desc=f"Downloading {split}"))

    print(f"Finished '{split}'. Successfully downloaded {sum(results)}/{len(tasks)} images.\n")

if __name__ == '__main__':
    # 1. Validation first (fast test)
    download_and_cache_dataset(split="validation", max_workers=128)
    # 2. Train second (long process)
    download_and_cache_dataset(split="train", max_workers=128)

Loading 'validation' dataset from Hugging Face...
Saving ALL annotations immediately to conceptual_captions_data/validation.jsonl...


Writing JSONL: 100%|██████████| 15840/15840 [00:00<00:00, 38676.96it/s]


Annotations saved. Starting download of 15840 images...


Finished 'validation'. Successfully downloaded 10601/15840 images.

Loading 'train' dataset from Hugging Face...
Saving ALL annotations immediately to conceptual_captions_data/train.jsonl...


Writing JSONL: 100%|██████████| 3318333/3318333 [01:24<00:00, 39063.04it/s]


Annotations saved. Starting download of 3318333 images...


  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(str(msg))
  warnings.warn(
  warnings.warn(str(msg))
  warnings.warn(
  warnings.warn(str(msg))
  warnings.warn(
  warnings.warn(
  warnings.warn(str(msg))
  warnings.warn(


Finished 'train'. Successfully downloaded 2167774/3318333 images.

